In [1]:
from pathlib import Path
import json
import shutil

In [5]:
# ============================================================
# НАСТРОЙКИ
# ============================================================

# Jupyter уже запущен из:
# C:\Users\Ева\Desktop\Хакатон\data_preproc
#
# Поэтому "." означает именно папку data_preproc.

ROOT_DIR = Path(".")

# Исходный COCO-датасет
COCO_DIR = ROOT_DIR / "Heavy Equipment_coco"

# Новый LabelMe-датасет
LABELME_DIR = ROOT_DIR / "Heavy Equipment_labelme"

# Папки нового датасета
IMAGES_DIR = LABELME_DIR / "images"
LABELS_DIR = LABELME_DIR / "labels"


# ============================================================
# ПРОВЕРКА ПУТЕЙ
# ============================================================

print("Текущая рабочая директория:")
print(Path.cwd().resolve())

print("\nИсходный датасет:")
print(COCO_DIR.resolve())

print("\nНовый датасет:")
print(LABELME_DIR.resolve())


if not COCO_DIR.exists():
    raise RuntimeError(
        f"\nИсходный датасет не найден:\n"
        f"{COCO_DIR.resolve()}\n\n"
        f"Проверь название папки CCTV_coco."
    )


# ============================================================
# СОЗДАНИЕ ПАПОК
# ============================================================

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# ПОИСК COCO JSON
# ============================================================

def find_coco_json(split_dir):

    json_files = list(split_dir.glob("*.json"))

    for json_file in json_files:

        try:

            with open(
                json_file,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

            if all(
                key in data
                for key in [
                    "images",
                    "annotations",
                    "categories"
                ]
            ):

                return json_file

        except Exception:
            continue

    return None


# ============================================================
# ПОИСК SPLIT-ПАПОК
# ============================================================

split_dirs = [
    folder
    for folder in COCO_DIR.iterdir()
    if folder.is_dir()
]


if not split_dirs:

    raise RuntimeError(
        f"\nВ папке CCTV_coco не найдено "
        f"ни одной подпапки:\n"
        f"{COCO_DIR.resolve()}"
    )


print("\nНайдены split-папки:")

for folder in split_dirs:
    print(f"  - {folder.name}")


# ============================================================
# ИМЕНА ИЗОБРАЖЕНИЙ
# ============================================================

used_image_names = set()


# ============================================================
# COCO -> LABELME
# ============================================================

def convert_split(
    split_dir,
    coco_data,
    split_name
):

    images = {
        image["id"]: image
        for image in coco_data["images"]
    }

    categories = {
        category["id"]: category["name"]
        for category in coco_data["categories"]
    }

    # --------------------------------------------------------
    # Группируем annotations по image_id
    # --------------------------------------------------------

    annotations_by_image = {}

    for annotation in coco_data["annotations"]:

        image_id = annotation["image_id"]

        annotations_by_image.setdefault(
            image_id,
            []
        ).append(annotation)

    # --------------------------------------------------------
    # Счетчики
    # --------------------------------------------------------

    copied_count = 0
    skipped_count = 0

    # --------------------------------------------------------
    # Обработка изображений
    # --------------------------------------------------------

    for image_id, image_info in images.items():

        original_filename = image_info["file_name"]

        source_image = split_dir / original_filename

        if not source_image.exists():

            print(
                f"[WARNING] Изображение не найдено:\n"
                f"    {source_image}"
            )

            skipped_count += 1
            continue

        # ====================================================
        # Определяем имя выходного файла
        # ====================================================

        output_filename = original_filename

        # Если такое имя уже существует,
        # добавляем имя split
        if (
            output_filename in used_image_names
            or (IMAGES_DIR / output_filename).exists()
        ):

            stem = Path(original_filename).stem
            suffix = Path(original_filename).suffix

            output_filename = (
                f"{split_name}_{stem}{suffix}"
            )

        # Если и это имя занято,
        # добавляем номер
        counter = 1

        while (
            output_filename in used_image_names
            or (IMAGES_DIR / output_filename).exists()
        ):

            stem = Path(original_filename).stem
            suffix = Path(original_filename).suffix

            output_filename = (
                f"{split_name}_{stem}_{counter}{suffix}"
            )

            counter += 1

        used_image_names.add(output_filename)

        # ====================================================
        # Копируем изображение
        # ====================================================

        destination_image = (
            IMAGES_DIR / output_filename
        )

        shutil.copy2(
            source_image,
            destination_image
        )

        copied_count += 1

        # ====================================================
        # Создаем shapes LabelMe
        # ====================================================

        labelme_shapes = []

        annotations = annotations_by_image.get(
            image_id,
            []
        )

        for annotation in annotations:

            category_id = annotation["category_id"]

            label = categories.get(
                category_id,
                str(category_id)
            )

            segmentation = annotation.get(
                "segmentation"
            )

            # =================================================
            # SEGMENTATION
            # =================================================

            if segmentation:

                if isinstance(
                    segmentation[0],
                    list
                ):

                    polygons = segmentation

                else:

                    polygons = [
                        segmentation
                    ]

                for polygon in polygons:

                    if len(polygon) < 6:
                        continue

                    points = []

                    for i in range(
                        0,
                        len(polygon),
                        2
                    ):

                        x = float(polygon[i])
                        y = float(polygon[i + 1])

                        points.append([
                            x,
                            y
                        ])

                    labelme_shapes.append({

                        "label": label,

                        "points": points,

                        "group_id": None,

                        "description": "",

                        "shape_type": "polygon",

                        "flags": {}

                    })

            # =================================================
            # BBOX
            # =================================================

            else:

                bbox = annotation.get("bbox")

                if (
                    bbox is not None
                    and len(bbox) == 4
                ):

                    x, y, width, height = bbox

                    x1 = float(x)
                    y1 = float(y)

                    x2 = float(x + width)
                    y2 = float(y + height)

                    labelme_shapes.append({

                        "label": label,

                        "points": [
                            [x1, y1],
                            [x2, y1],
                            [x2, y2],
                            [x1, y2]
                        ],

                        "group_id": None,

                        "description": "",

                        "shape_type": "polygon",

                        "flags": {}

                    })

        # ====================================================
        # LabelMe JSON
        # ====================================================

        labelme_data = {

            "version": "5.4.1",

            "flags": {},

            "shapes": labelme_shapes,

            "imagePath": f"../images/{output_filename}",

            "imageData": None,

            "imageHeight": image_info["height"],

            "imageWidth": image_info["width"]

        }

        # ====================================================
        # СОХРАНЯЕМ JSON
        # ====================================================

        label_filename = (
            Path(output_filename).stem + ".json"
        )

        output_label = (
            LABELS_DIR / label_filename
        )

        with open(
            output_label,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                labelme_data,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(
            f"[OK] {split_name:<8} "
            f"{original_filename} -> "
            f"{output_filename}"
        )

    return copied_count, skipped_count


# ============================================================
# ОБРАБОТКА ВСЕХ SPLIT
# ============================================================

processed_splits = []
skipped_splits = []

total_images = 0
total_skipped = 0


for split_dir in split_dirs:

    split_name = split_dir.name

    coco_json = find_coco_json(split_dir)

    if coco_json is None:

        skipped_splits.append(split_name)

        print(
            f"\n[SKIP] {split_name}: "
            f"COCO JSON не найден"
        )

        continue

    print("\n" + "=" * 70)

    print(
        f"Обработка split: {split_name}"
    )

    print(
        f"COCO JSON: {coco_json.name}"
    )

    print("=" * 70)

    with open(
        coco_json,
        "r",
        encoding="utf-8"
    ) as f:

        coco_data = json.load(f)

    copied, skipped = convert_split(
        split_dir=split_dir,
        coco_data=coco_data,
        split_name=split_name
    )

    total_images += copied
    total_skipped += skipped

    processed_splits.append(split_name)


# ============================================================
# ИТОГ
# ============================================================

print("\n" + "=" * 70)
print("ГОТОВО")
print("=" * 70)

print(
    f"\nИсходный датасет:\n"
    f"  {COCO_DIR.resolve()}"
)

print(
    f"\nНовый LabelMe:\n"
    f"  {LABELME_DIR.resolve()}"
)

print(
    f"\nИзображений скопировано: {total_images}"
)

print(
    f"Изображений пропущено:   {total_skipped}"
)

print("\nОбработанные split:")

for split in processed_splits:
    print(f"  ✓ {split}")

if skipped_splits:

    print("\nПапки без COCO JSON:")

    for split in skipped_splits:
        print(f"  - {split}")

Текущая рабочая директория:
C:\Users\Ева\Desktop\Хакатон\data_preproc

Исходный датасет:
C:\Users\Ева\Desktop\Хакатон\data_preproc\Heavy Equipment_coco

Новый датасет:
C:\Users\Ева\Desktop\Хакатон\data_preproc\Heavy Equipment_labelme

Найдены split-папки:
  - train

Обработка split: train
COCO JSON: _annotations.coco.json
[OK] train    vlcsnap-2024-07-08-13h14m14s323_png.rf.7e09a6e660dc6a3824966db32acb52b8.jpg -> vlcsnap-2024-07-08-13h14m14s323_png.rf.7e09a6e660dc6a3824966db32acb52b8.jpg
[OK] train    vlcsnap-2024-07-08-13h08m05s647_png.rf.7e2e6abb78151d749d606e4894bde315.jpg -> vlcsnap-2024-07-08-13h08m05s647_png.rf.7e2e6abb78151d749d606e4894bde315.jpg
[OK] train    vlcsnap-2024-07-08-13h20m12s615_png.rf.7e111237a5a56c95fbfc49b567cd3ce3.jpg -> vlcsnap-2024-07-08-13h20m12s615_png.rf.7e111237a5a56c95fbfc49b567cd3ce3.jpg
[OK] train    vlcsnap-2024-07-01-18h17m24s827_png.rf.7e58ad06e219e241776f855e259a1185.jpg -> vlcsnap-2024-07-01-18h17m24s827_png.rf.7e58ad06e219e241776f855e259a1185.jpg


In [4]:
from pathlib import Path
import json


# ============================================================
# НАСТРОЙКИ
# ============================================================

# Папка с LabelMe JSON
LABELS_DIR = Path(r"CCTV_labelme/labels")


# ============================================================
# СООТВЕТСТВИЕ КЛАССОВ
# ============================================================

# Здесь указываем:
# "старое название": "новое название"
#
# Можно использовать любое количество классов.

CLASS_MAPPING = {
    "0": "бетономешалка",
    "1": "грузовик",
    "2": "экскаватор",
}


# ============================================================
# ПРОВЕРКА
# ============================================================

if not LABELS_DIR.exists():
    raise RuntimeError(
        f"Папка с LabelMe JSON не найдена:\n"
        f"{LABELS_DIR.resolve()}"
    )

print("Папка с аннотациями:")
print(LABELS_DIR.resolve())

print("\nСоответствие классов:")

for old_name, new_name in CLASS_MAPPING.items():
    print(f"  {old_name} -> {new_name}")


# ============================================================
# ПЕРЕИМЕНОВАНИЕ КЛАССОВ
# ============================================================

json_files = list(LABELS_DIR.glob("*.json"))

if not json_files:
    print("\nJSON-файлы не найдены.")
else:

    changed_json = 0
    changed_labels = 0
    unknown_labels = set()

    for json_file in json_files:

        # ----------------------------------------------------
        # Читаем JSON
        # ----------------------------------------------------

        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        file_changed = False

        # ----------------------------------------------------
        # Обрабатываем shapes
        # ----------------------------------------------------

        for shape in data.get("shapes", []):

            old_label = shape.get("label")

            if old_label in CLASS_MAPPING:

                new_label = CLASS_MAPPING[old_label]

                if old_label != new_label:

                    shape["label"] = new_label

                    changed_labels += 1
                    file_changed = True

            else:

                # Запоминаем классы,
                # которых нет в словаре
                if old_label is not None:
                    unknown_labels.add(old_label)

        # ----------------------------------------------------
        # Сохраняем JSON
        # ----------------------------------------------------

        if file_changed:

            with open(
                json_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    data,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            changed_json += 1

            print(
                f"[OK] {json_file.name}"
            )


# ============================================================
# ИТОГ
# ============================================================

print("\n" + "=" * 70)
print("ГОТОВО")
print("=" * 70)

print(
    f"\nОбработано JSON: "
    f"{len(json_files)}"
)

print(
    f"Изменено JSON: "
    f"{changed_json}"
)

print(
    f"Изменено объектов: "
    f"{changed_labels}"
)

if unknown_labels:

    print(
        "\nВНИМАНИЕ! Найдены классы, "
        "которых нет в CLASS_MAPPING:"
    )

    for label in sorted(
        unknown_labels,
        key=str
    ):
        print(f"  - {label}")

else:

    print(
        "\nВсе найденные классы "
        "есть в CLASS_MAPPING."
    )

Папка с аннотациями:
C:\Users\Ева\Desktop\Хакатон\data_preproc\CCTV_labelme\labels

Соответствие классов:
  0 -> бетономешалка
  1 -> грузовик
  2 -> экскаватор
[OK] 000001_jpg.rf.758184e127cbc907e8c8cc8dd9974125.json
[OK] 000002_jpg.rf.626da3bdb96939a56d459e81100c4f20.json
[OK] 000003_jpg.rf.08f9c3e9904a0fd81423b998eb8296be.json
[OK] 000004_jpg.rf.028897721b46d58d12918685d78cadaf.json
[OK] 000005_jpg.rf.1a09355b2d3a2be217107e830e72bb87.json
[OK] 000006_jpg.rf.2e7bbffdf578b480f0491e15a3327a42.json
[OK] 000007_jpg.rf.f45cb224610eddcc679ab6acc8b191f2.json
[OK] 000008_jpg.rf.c5e43a3b652fbeb3c93408b03cfc82d4.json
[OK] 000009_jpg.rf.4295b97e8dec25cc53b932e2ece01ec7.json
[OK] 000010_jpg.rf.762da09d09e78db09092f3f5cee88288.json
[OK] 000011_jpg.rf.5dc981d2c2361dc75512a587bbcb0a54.json
[OK] 000012_jpg.rf.64bdc4bd12dd20543fa51c5f04c6b255.json
[OK] 000013_jpg.rf.5c709dba4ce1b54fe016dacdd9ae516f.json
[OK] 000014_jpg.rf.80c12ad7fdd9d72be8502c420a832396.json
[OK] 000015_jpg.rf.5c6d9bd880824d50272266